# 02 Analysis
Answers Q1 - Q3 using cleaned data. 

Weighted results with 95% CIs

Small cell rule: n < 30 don't report, 30-50 report with caution

## 1. Load

In [1]:
import sys
import pandas as pd
sys.path.append("..")
from src.helpers import weighted_pct

# Load cleaned data files
df = pd.read_csv("../data/processed/clean.csv").copy()
tasks = pd.read_csv("../data/processed/tasks.csv").copy()

# Check
print("clean csv:", df.shape)
print("tasks.csv", tasks.shape)

# Check 2
key_cols = ["weight", "age_group", "ai_work", "net_saver", "bls_exposure", "bls_conf"]
print(df[key_cols].dtypes, "\n")
print("Weighted % using AI at work:", round(weighted_pct(df, "ai_work"), 1))

clean csv: (1103, 348)
tasks.csv (798, 14)
weight          float64
age_group           str
ai_work         float64
net_saver       float64
bls_exposure        str
bls_conf        float64
dtype: object 

Weighted % using AI at work: 42.9


## 2. CI method
95% CIs use Kish's efective sample size(n_eff) + normal formula

Limitation: public file hasno strata/cluster info, so CIs are approximate

In [2]:
from src.helpers import weighted_ci

# Test on overall number we already know (42.9%)
result = weighted_ci(df, "ai_work")
for k, v in result.items():
    print(f"{k:6} {v:.2f}")

pct    42.89
se     1.50
ci_lo  39.94
ci_hi  45.83
n      1102.00
n_eff  1086.65


In [3]:
# Epoch's published summary table
summary = pd.read_csv("../data/raw/polling_on_ai_usage_jul_2026.csv")
s = summary[(summary["pct"] > 1) & (summary["pct"] < 99)].copy()

# Work backwards: design effect = variance/ plain random samlpe variance
p = s["pct"] / 100
s["implied_deff"] = s["n_base"] * (s["se"]/ 100) ** 2/ (p*(1-p))

print(s[["demographic", "demographic_value", "n_base", "pct", "se", "implied_deff"]].head(8), "\n")
print(s["implied_deff"].describe())

   demographic demographic_value  n_base    pct    se  implied_deff
0  full_sample               All    1103  29.02  1.37      1.005040
1  full_sample               All    1103  19.36  1.19      1.000492
2  full_sample               All    1103  31.41  1.40      1.003466
3  full_sample               All    1103  13.04  1.01      0.992250
4  full_sample               All    1103  29.47  1.38      1.010601
5  full_sample               All    1103   8.73  0.85      1.000164
6  full_sample               All    1103  15.72  1.09      0.989126
7  full_sample               All    1103  31.15  1.40      1.008021 

count    2928.000000
mean        1.345142
std         0.625220
min         0.824002
25%         1.011862
50%         1.034861
75%         1.296093
max         4.968402
Name: implied_deff, dtype: float64


## 3. Q1: Is it age, or something else?
### 3a. AI use at work by age (baseline, reproduces Epoch)

In [4]:
rows = []

for group in sorted(df["age_group"].dropna().unique()):
    sub = df[df["age_group"] == group]
    r = weighted_ci(sub, "ai_work")
    r["age_group"] = group
    rows.append(r)

q1a = pd.DataFrame(rows).set_index("age_group").round(1)
print(q1a)

            pct   se  ci_lo  ci_hi    n  n_eff
age_group                                     
18-29      42.0  3.3   35.6   48.4  232  229.2
30-44      49.1  2.6   44.1   54.1  381  378.4
45-59      41.2  2.8   35.7   46.7  312  310.3
60+        32.3  3.5   25.4   39.2  177  175.9


### 3b. Age x College (is the age pattern really about degrees?)

In [5]:
# How many people (unweighted) in each age x college cell
print(pd.crosstab(df["age_group"], df["college"], margins=True))

college    False  True   All
age_group                   
18-29        174    58   232
30-44        192   190   382
45-59        162   150   312
60+          105    72   177
All          633   470  1103


In [6]:
rows = []

for deg in[False, True]:
    for group in sorted(df["age_group"].dropna().unique()):
        sub = df[(df["age_group"] == group) & (df["college"] == deg)]
        r = weighted_ci(sub, "ai_work")
        r["college"] = "Degree" if deg else "No degree"
        r["age_group"] = group
        rows.append(r)

q1b = pd.DataFrame(rows).set_index(["college", "age_group"]).round(1)
print(q1b[["pct", "ci_lo", "ci_hi", "n"]])

                      pct  ci_lo  ci_hi    n
college   age_group                         
No degree 18-29      38.7   31.4   46.0  174
          30-44      41.9   34.9   48.9  192
          45-59      32.3   25.1   39.5  162
          60+        24.5   16.3   32.8  105
Degree    18-29      52.7   39.8   65.7   58
          30-44      56.8   49.7   63.9  189
          45-59      51.3   43.3   59.3  150
          60+        44.2   32.7   55.6   72


### 3c. Age x job type(is age pattern really about the kind of job?)

In [7]:
# People(unweighted) in each age x job type cell
print(pd.crosstab(df["age_group"], df["job_broad"].fillna("No job group"), margins=True))

job_broad  Care & education  Frontline  No job group  Office/knowledge   All
age_group                                                                   
18-29                    43        103            23                63   232
30-44                    68        103            39               172   382
45-59                    57         92            33               130   312
60+                      28         56            25                68   177
All                     196        354           120               433  1103


In [8]:
rows = []
job_types = ["Office/knowledge", "Care & education", "Frontline"]

for job in job_types:
    for group in sorted(df["age_group"].dropna().unique()):
        sub = df[(df["age_group"] == group) & (df["job_broad"] == job)]
        r = weighted_ci(sub, "ai_work")
        r["job_broad"] = job
        r["age_group"] = group
        rows.append(r)

q1b_job = pd.DataFrame(rows).set_index(["job_broad", "age_group"]).round(1)

# Small-cell rule (decision A): blank out n < 30, flag 30-50
q1b_job.loc[q1b_job["n"] < 30, ["pct", "ci_lo", "ci_hi"]] = None
q1b_job["note"] = ""
q1b_job.loc[q1b_job["n"] < 30, "note"] = "not reported (n<30)"
q1b_job.loc[(q1b_job["n"] >= 30) & (q1b_job["n"] < 50), "note"] = "caution (n 30-50)"

print(q1b_job[["pct", "ci_lo", "ci_hi", "n", "note"]])

                             pct  ci_lo  ci_hi    n                 note
job_broad        age_group                                              
Office/knowledge 18-29      68.6   57.0   80.1   63                     
                 30-44      59.6   52.2   67.0  171                     
                 45-59      51.2   42.6   59.8  130                     
                 60+        50.0   38.1   61.9   68                     
Care & education 18-29      38.6   23.9   53.2   43    caution (n 30-50)
                 30-44      47.0   35.1   58.9   68                     
                 45-59      42.8   30.0   55.7   57                     
                 60+         NaN    NaN    NaN   28  not reported (n<30)
Frontline        18-29      28.3   19.6   37.1  103                     
                 30-44      39.8   30.3   49.3  103                     
                 45-59      30.3   20.9   39.7   92                     
                 60+        20.2    9.7   30.8   56

### 3d. Logistic Regression: warm up(age only)
Survey weighted logistic regression with robust (HC1) standard errors

with age as the only predictor, predicted % should equal the Q1A table

In [9]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Rows hte model can use: outcome and age
d = df.dropna(subset=["ai_work", "age_group"]).copy()

m0 = smf.glm(
    "ai_work ~ C(age_group, Treatment('18-29'))",
    data = d,
    family = sm.families.Binomial(),
    var_weights = d["weight"],
).fit(cov_type="HC1")

print(m0.summary())

# Check
d["pred"] = m0.predict(d)
print((d.groupby("age_group")["pred"].first()* 100).round(1))

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ai_work   No. Observations:                 1102
Model:                            GLM   Df Residuals:                     1098
Model Family:                Binomial   Df Model:                            3
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -745.85
Date:                Wed, 21 Oct 2026   Deviance:                       1491.7
Time:                        15:30:37   Pearson chi2:                 1.10e+03
No. Iterations:                     4   Pseudo R-squ. (CS):            0.01238
Covariance Type:                  HC1                                         
                                                coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------

### 3e. Model 1: age + degree + job type + controls
Decisions: all controls (1A) · age vs 18-29 (2A) · "No job group" kept as its own category (3B).


In [10]:
model_vars = ["ai_work", "age_group", "college", "job_broad",
              "gender", "income_reg", "race", "weight"]

dm = df[model_vars].copy()
dm["job_broad"] = dm["job_broad"].fillna("No job group")

# Blanks per variable
print(dm.isna().sum(), "\n")

# How many poeple model would keep
print("Rows before: ", len(dm))
print("Rows complete:", len(dm.dropna()))


ai_work       1
age_group     0
college       0
job_broad     0
gender        0
income_reg    0
race          0
weight        0
dtype: int64 

Rows before:  1103
Rows complete: 1102


In [11]:
dm = dm.dropna() 

m1 = smf.glm(
    "ai_work ~ C(age_group, Treatment('18-29'))"
    " + C(college)"
    " + C(job_broad, Treatment('Office/knowledge'))"
    " + C(gender) + C(income_reg) + C(race)",
    data=dm,
    family=sm.families.Binomial(),
    var_weights = dm["weight"],
).fit(cov_type="HC1")

print(m1.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ai_work   No. Observations:                 1102
Model:                            GLM   Df Residuals:                     1085
Model Family:                Binomial   Df Model:                           16
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -691.98
Date:                Wed, 21 Oct 2026   Deviance:                       1384.0
Time:                        15:30:37   Pearson chi2:                 1.10e+03
No. Iterations:                     4   Pseudo R-squ. (CS):             0.1044
Covariance Type:                  HC1                                         
                                                                      coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------